# Launch test ABC

# Distribution of fitness effects

## Load tree and metadata

In [ ]:
import numpy as np
import pandas as pd
from ete3 import Tree

tree = Tree("./data/subsampled_tree.rooted.nex", format=1)
tip_names = sorted(tree.get_leaf_names())

metadata_path = "./data/metadata_reduced.tsv"

# Create lineage index
metadata = pd.read_csv(metadata_path, sep='\t')
lineage_map = dict(zip(metadata['GNUMBER'], metadata['LINEAGE_x']))

# Drop keys that are not in the tree
lineage_map = {k: v for k, v in lineage_map.items() if k in tip_names}

# Assign index to each lineage
unique_lineages = list(set(lineage_map.values()))
lineage_idx = [unique_lineages.index(lineage_map[name]) for name in tip_names]

print(unique_lineages)

## Visualize distributions

### Priors
Priors for five parameters of the model are specified in abc_priors.yaml

- p_neutral
- gamma_shape
- gamma_scale
- r_birth
- r_loss

In [ ]:
import yaml
import numpy as np
import seaborn as sns

from math import ceil
from lib import helpers

import matplotlib.pyplot as plt

yaml_path = "./abc_priors.yaml"
with open(yaml_path) as f:
    priors = yaml.safe_load(f)

# Show the loaded priors
params = ['p_neutral', 'gamma_shape', 'gamma_scale', 'r_birth', 'r_loss']

print("Loaded priors from", yaml_path)
for k, v in priors.items():
    if k in params:
        print(f" - {k}: {v}")


Plot prior distributions.

In [ ]:
# Plotting
keys = params
n = len(keys)
cols = 3
rows = ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
axes = axes.flatten()

for i, key in enumerate(keys):
    spec = priors[key]
    try:
        samples = helpers.sample_from_spec(spec, size=20000)
    except Exception as e:
        axes[i].text(0.5, 0.5, f"Error parsing prior:\n{e}", ha="center", va="center")
        axes[i].set_title(key)
        axes[i].set_xticks([])
        axes[i].set_yticks([])
        continue

    # If categorical (non-numeric), plot counts
    if not np.issubdtype(np.array(samples).dtype, np.number):
        sns.countplot(x=samples, ax=axes[i])
        axes[i].set_xticklabels(axes[i].get_xticklabels(), rotation=45, ha="right")
    else:
        sns.histplot(samples, kde=True, stat="density", ax=axes[i], bins=60, color="#2b8cbe")
    axes[i].set_title(key)

# Turn off any unused axes
for j in range(n, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

### Fitness effects

In [ ]:
from scipy.stats import gamma

import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

params = [
    (priors['gamma_shape']['lower'], priors['gamma_scale']['lower']),
    (priors['gamma_shape']['lower'], priors['gamma_scale']['upper']),
    (priors['gamma_shape']['upper'], priors['gamma_scale']['upper']),
    (priors['gamma_shape']['upper'], priors['gamma_scale']['lower']),
]
titles = [
    f"shape={params[0][0]:.2f}, scale={params[0][1]:.2f}",
    f"shape={params[1][0]:.2f}, scale={params[1][1]:.2f}",
    f"shape={params[2][0]:.2f}, scale={params[2][1]:.2f}",
    f"shape={params[3][0]:.2f}, scale={params[3][1]:.2f}",
]

for idx, (shape, scale) in enumerate(params):
    ax = axes.flatten()[idx]
    x = np.linspace(0, gamma.ppf(0.99, shape, scale=scale), 1000)
    y = gamma.pdf(x, shape, scale=scale)
    
    ax.plot(x, y, linewidth=2, color='#2b8cbe')
    ax.fill_between(x, y, alpha=0.3, color='#2b8cbe')
    ax.set_title(titles[idx], fontsize=12)
    ax.set_xlabel('Fitness effect')
    ax.set_ylabel('Probability density')
    ax.grid(True, alpha=0.3)

fig.suptitle('Gamma DFE Distribution Extremes', fontsize=14)
plt.tight_layout()
plt.show()

### Genomic niches

In [ ]:

from niche_model_DFE import create_niche_space, create_targetability

L=7178
# number of prior samples to draw
n = 1

# parameters to sample from priors
param_keys = ["p_neutral", "gamma_shape", "gamma_scale", "r_birth", "r_loss"]
samples = {k: helpers.sample_from_spec(priors[k], size=n) for k in param_keys}


occ, fitness = create_niche_space(
    L, samples['p_neutral'], samples['gamma_shape'], samples['gamma_scale'], initial_copies=1)

targetability = create_targetability(L, shape=2.0)

## Run some simulations

In [ ]:
n = 100

# parameters to sample from priors
param_keys = ["p_neutral", "gamma_shape", "gamma_scale", "r_birth", "r_loss"]
samples = {k: helpers.sample_from_spec(priors[k], size=n) for k in param_keys}

In [ ]:
from niche_model_DFE import run_simulation

results = []
for i in range(n):
    params_kwargs = {k: float(samples[k][i]) for k in param_keys}
    print(i, params_kwargs)
    try:
        params_out, cn_out, stats_out = run_simulation(
            tree=tree,
            L=7178,
            tip_names=tip_names,
            lineage_map=lineage_map,
            unique_lineages=unique_lineages,
            seed=i,
            **params_kwargs
        )
        row = params_kwargs.copy()
        row.update(stats_out)
    except Exception as e:
        row = params_kwargs.copy()
        row.update({"error": str(e)})
    results.append(row)

results_df = pd.DataFrame(results)
results_df.to_csv("prior_simulations.csv", index=False)
results_df.head()

## Check how parameters affect summary stats

... using random forests.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
import pandas as pd

import matplotlib.pyplot as plt

# Load the results
results_df = pd.read_csv("prior_simulations.csv")

results_df.head()

In [ ]:

# Define parameters and statistics
params = ['p_neutral', 'gamma_shape', 'gamma_scale', 'r_birth', 'r_loss']
stats = [col for col in results_df.columns if col not in params]
print(f'Number of summary stats: {len(stats)}')

# Train random forests for each statistic
feature_importances = {}
for stat in stats:
    X = results_df[params]
    y = results_df[stat]
    
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X, y)
    feature_importances[stat] = dict(zip(params, rf.feature_importances_))

In [ ]:
# Visualize feature importances
fig, axes = plt.subplots(4, 5, figsize=(16, 14))
axes = axes.flatten()

for idx, (stat, importances) in enumerate(feature_importances.items()):
    if idx >= len(axes):
        break
    ax = axes[idx]
    
    params_list = list(importances.keys())
    values = list(importances.values())
    
    ax.barh(params_list, values, color='#2b8cbe')
    ax.set_xlabel('Feature Importance')
    ax.set_title(f'Feature Importance for {stat}')
    ax.set_xlim(0, max(values) * 1.1)

# Turn off unused axes
for j in range(len(feature_importances), len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

# Simpler model with p_essential and p_tolerated

## Single simulation

In [ ]:
import numpy as np
import pandas as pd
from niche_model_numba import run_simulation
from ete3 import Tree

tree = Tree("../data/subsampled_tree.rooted.nex", format=1)
tip_names = sorted(tree.get_leaf_names())

metadata_path = "../data/metadata_reduced.tsv"

 # Create lineage index
metadata = pd.read_csv(metadata_path, sep='\t')
lineage_map = dict(zip(metadata['GNUMBER'], metadata['LINEAGE_x']))

# Assign index to each lineage
unique_lineages = list(set(lineage_map.values()))
lineage_idx = [unique_lineages.index(lineage_map[name]) for name in tip_names]

print(unique_lineages)


In [ ]:
params, cn, stats = run_simulation(
    tree=tree,
    L=7178,
    p_essential=0.5,
    p_tolerated=0.4,
    r_birth=0.0001,
    r_purge=0.001,
    tip_names=tip_names,
    lineage_idx=lineage_idx,
    unique_lineages=unique_lineages
)
stats

In [ ]:
import matplotlib.pyplot as plt

param_grid = {
    "p_essential": [0.65, 0.70, 0.75],
    "p_tolerated": [0.25, 0.28],
    "r_birth": [1e-5, 5e-5],
    "r_purge": [1e-4, 3e-4],
}
replicates = 2

results = []
for p_essential in param_grid["p_essential"]:
    for p_tolerated in param_grid["p_tolerated"]:
        if p_essential + p_tolerated >= 1.0:
            continue
        for r_birth in param_grid["r_birth"]:
            for r_purge in param_grid["r_purge"]:
                for rep in range(replicates):
                    _, _, stats_sim = run_simulation(
                        tree=tree,
                        L=8000,
                        p_essential=p_essential,
                        p_tolerated=p_tolerated,
                        r_birth=r_birth,
                        r_purge=r_purge,
                        tip_names=tip_names,
                        lineage_idx=np.zeros(len(tip_names), dtype=int),
                        unique_lineages=["all"],
                        seed=43 + rep
                    )
                    results.append({
                        "p_essential": p_essential,
                        "p_tolerated": p_tolerated,
                        "r_birth": r_birth,
                        "r_purge": r_purge,
                        "replicate": rep,
                        "mean_cn": stats_sim["mean_cn"],
                        "std_cn": stats_sim["std_cn"],
                        "gini_occupancy": stats_sim["gini_occupancy"],
                        "tajimas_d": stats_sim["tajimas_d"],
                        "n_singletons": stats_sim["n_singletons"],
                    })

p_ess_values = sorted({r["p_essential"] for r in results})
p_tol_values = sorted({r["p_tolerated"] for r in results})
r_birth_values = sorted({r["r_birth"] for r in results})
r_purge_values = sorted({r["r_purge"] for r in results})

mean_cn_by_pe = [[r["mean_cn"] for r in results if r["p_essential"] == pe] for pe in p_ess_values]
tajimas_by_pt = [[r["tajimas_d"] for r in results if r["p_tolerated"] == pt] for pt in p_tol_values]
gini_by_rb = [[r["gini_occupancy"] for r in results if r["r_birth"] == rb] for rb in r_birth_values]
singletons_by_rp = [[r["n_singletons"] for r in results if r["r_purge"] == rp] for rp in r_purge_values]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

axes[0, 0].boxplot(mean_cn_by_pe, labels=p_ess_values)
axes[0, 0].set_title("mean_cn by p_essential")
axes[0, 0].set_xlabel("p_essential")
axes[0, 0].set_ylabel("mean copy number")

axes[0, 1].boxplot(tajimas_by_pt, labels=p_tol_values)
axes[0, 1].set_title("tajimas_d by p_tolerated")
axes[0, 1].set_xlabel("p_tolerated")
axes[0, 1].set_ylabel("Tajima's D")

axes[1, 0].boxplot(gini_by_rb, labels=[f"{rb:.0e}" for rb in r_birth_values])
axes[1, 0].set_title("gini_occupancy by r_birth")
axes[1, 0].set_xlabel("r_birth")
axes[1, 0].set_ylabel("Gini occupancy")

axes[1, 1].boxplot(singletons_by_rp, labels=[f"{rp:.0e}" for rp in r_purge_values])
axes[1, 1].set_title("n_singletons by r_purge")
axes[1, 1].set_xlabel("r_purge")
axes[1, 1].set_ylabel("Number of singletons")

fig.suptitle("Simulation summary stat distributions across parameter ranges", fontsize=16)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()